In [ ]:
from fyers_apiv3 import fyersModel

# Your App Credentials
client_id = "" # E.g., ABCD1234-100
secret_key = ""
redirect_uri = "https://127.0.0.1:8080" # Must match exactly what you put in the dashboard

# Step 1: Generate the Login URL
session = fyersModel.SessionModel(
    client_id=client_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type="code",
    grant_type="authorization_code"
)
login_url = session.generate_authcode()
print("Click this URL to log in and authorize:", login_url)

# Step 2: After logging in, you will be redirected to an empty page. 
# Look at the URL in your browser. It will look like: 
# https://127.0.0.1:8080/?auth_code=XXXXXXXXXXX&state=None
# Copy the XXXXXXXXXXX part.

auth_code = input("Enter the auth_code from the URL: ")

# Step 3: Generate the Access Token
session.set_token(auth_code)
response = session.generate_token()
access_token = response['access_token']

print("Success! Your Access Token for today is:")
print(access_token)

In [ ]:
from fyers_apiv3 import fyersModel
import pandas as pd
import datetime
import time

# Initialize the Fyers Model (Ensure client_id and access_token are defined above this)
fyers = fyersModel.FyersModel(client_id=client_id, is_async=False, token=access_token, log_path="")

# 1. Define Fyers-specific symbols for the indices
symbols = {
    "NIFTY_50": "NSE:NIFTY50-INDEX",
    "NIFTY_MIDCAP_150": "NSE:NIFTYMIDCAP150-INDEX",
    "NIFTY_SMALLCAP_250": "NSE:NIFTYSMLCAP250-INDEX"
}

# 2. Set the overall date range (April 1st, 2016 to Today)
start_date = datetime.date(2016, 4, 1)
end_date = datetime.date.today()

# 3. Helper function to fetch data in 365-day chunks
def fetch_historical_data(symbol, start, end):
    all_candles = []
    current_start = start
    
    while current_start <= end:
        # Fyers API limit for 'D' resolution is 365 days per request
        current_end = current_start + datetime.timedelta(days=364)
        if current_end > end:
            current_end = end
            
        data_params = {
            "symbol": symbol,
            "resolution": "D",
            "date_format": "1", # YYYY-MM-DD format
            "range_from": current_start.strftime("%Y-%m-%d"),
            "range_to": current_end.strftime("%Y-%m-%d"),
            "cont_flag": "1"
        }
        
        response = fyers.history(data=data_params)
        
        if response['s'] == "ok":
            all_candles.extend(response['candles'])
            print(f"  -> Fetched {current_start} to {current_end}")
        else:
            print(f"  -> Error fetching chunk {current_start} to {current_end}: {response}")
            
        # Move start date to the day after the current chunk ends
        current_start = current_end + datetime.timedelta(days=1)
        
        # Sleep for 0.5 seconds to avoid hitting API rate limits
        time.sleep(0.5)
        
    return all_candles

# 4. Main execution loop
for name, ticker in symbols.items():
    print(f"\n========================================")
    print(f"Extracting {name} ({ticker})")
    print(f"========================================")
    
    # Fetch all chunks
    candles = fetch_historical_data(ticker, start_date, end_date)
    
    if not candles:
        print(f"No data retrieved for {name}. Skipping to next.")
        continue
        
    # Convert the stitched data into a Pandas DataFrame
    columns = ['Epoch_Time', 'Open', 'High', 'Low', 'Close', 'Volume']
    df = pd.DataFrame(candles, columns=columns)
    
    # Format the Date Index
    df['Date'] = pd.to_datetime(df['Epoch_Time'], unit='s')
    df.set_index('Date', inplace=True)
    df.drop('Epoch_Time', axis=1, inplace=True)
    
    # 5. Save to an independent CSV file
    filename = f"{name}_EOD_Features.csv"
    df.to_csv(filename)
    print(f"\nSUCCESS: Saved {len(df)} records to {filename}")

print("\nAll data extraction and feature engineering complete!")